# PBR 1倍割れ Research Pilot

**問い:** PBR 1倍割れは割安候補なのか、それとも低収益性・財務健全性・キャッシュ創出力で説明できる妥当なディスカウントなのか。

- H1: PBR 1倍割れ企業は、それ以外より収益性が低い
- H2: 低PBR群の中でも収益性・自己資本比率・営業CFには差がある
- H3: 低PBRかつ収益性・CFが相対的に良い企業は継続調査候補になり得る
- H4: 現行データだけではB/Sの質やカタリストを十分評価できない

このNotebookは探索成果物です。出力はAssessmentでもDecisionでもなく、`PBR < 1`を割安・買いとは判定しません。

In [9]:
import os
import subprocess
from datetime import UTC, datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from research.pbr_value_trap import (
    ResearchParameters,
    prepare_research_connection,
    quality_summary,
)

ROOT = Path.cwd()
if not (ROOT / 'research').is_dir():
    raise RuntimeError('リポジトリルートからNotebookを起動してください: make research-pbr')

LAKE = os.getenv('PARQUET_LAKE', str(ROOT / 'data' / 'parquet'))
AS_OF_DATE = os.getenv('RESEARCH_AS_OF_DATE', '2026-09-30')
PBR_THRESHOLD = float(os.getenv('RESEARCH_PBR_THRESHOLD', '1.0'))
STALE_AFTER_DAYS = int(os.getenv('RESEARCH_STALE_AFTER_DAYS', '550'))
params = ResearchParameters.parse(
    as_of_date=AS_OF_DATE,
    pbr_threshold=PBR_THRESHOLD,
    stale_after_days=STALE_AFTER_DAYS,
)
git_commit = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], cwd=ROOT, capture_output=True, text=True, check=True
).stdout.strip()
print({
    'lake': LAKE,
    'as_of_date': AS_OF_DATE,
    'pbr_threshold': PBR_THRESHOLD,
    'stale_after_days': STALE_AFTER_DAYS,
    'executed_at_utc': datetime.now(UTC).isoformat(),
    'git_commit': git_commit,
})
connection = prepare_research_connection(LAKE, params)

ModuleNotFoundError: No module named 'research'

## 1. Data contract / quality checks
将来情報を混ぜず、銘柄ごとに基準日以前の最終価格と、その時点で既に公表されていた財務を使います。

In [ ]:
quality = quality_summary(connection)
display(pd.DataFrame([quality]).T.rename(columns={0: 'value'}))
assert quality['universe_count'] > 0, '分析母集団が空です'
assert quality['calculable_count'] > 0, 'PBRを計算できる行がありません'
assert quality['duplicate_security_count'] == 0, 'security_keyの重複があります'
master_join_rate = 1 - quality['missing_security_master'] / quality['universe_count']
assert master_join_rate >= 0.90, f'銘柄マスタ結合率が低すぎます: {master_join_rate:.1%}'
print(f'security master join rate: {master_join_rate:.1%}')

In [ ]:
status = connection.execute(
    '''
    SELECT calculation_status, COUNT(*) AS rows
    FROM pbr_research_base
    GROUP BY calculation_status
    ORDER BY rows DESC
    '''
).df()
display(status)
point_in_time_violations = connection.execute(
    '''
    SELECT COUNT(*)
    FROM pbr_research_base
    WHERE financial_disclosed_date >= as_of_date
    '''
).fetchone()[0]
assert point_in_time_violations == 0, '同日または未来の財務開示が混入しています'

NameError: name 'connection' is not defined

## 2. Universe construction
計算可能な正のPBRを母集団とし、低PBR群と比較群を分けます。金融業は削除せず、業種別集計で差を確認します。

In [ ]:
universe = connection.execute(
    '''
    SELECT *
    FROM pbr_research_base
    WHERE calculation_status = 'ok' AND pbr > 0
    '''
).df()
universe['pbr_group'] = universe['is_below_pbr_threshold'].map(
    {True: 'below_threshold', False: 'at_or_above_threshold'}
)
display(universe.groupby('pbr_group', dropna=False).size().rename('rows').to_frame())

## 3. Cross-sectional analysis — H1 / H2
平均だけでなく中央値・四分位・件数・欠損を確認します。ROEとFCFは簡便なproxyです。

In [ ]:
metrics = ['roe_proxy', 'equity_ratio', 'operating_margin', 'free_cash_flow_proxy']
summary = universe.groupby('pbr_group')[metrics].agg(
    ['count', 'median', lambda values: values.quantile(0.25), lambda values: values.quantile(0.75)]
)
summary = summary.rename(columns={'<lambda_0>': 'q25', '<lambda_1>': 'q75'})
display(summary)
missing = universe.groupby('pbr_group')[metrics].agg(lambda values: values.isna().mean())
display(missing.style.format('{:.1%}'))

In [ ]:
sector = connection.execute(
    '''
    SELECT
        COALESCE(sector_33_name, 'Unknown') AS sector,
        COUNT(*) FILTER (WHERE calculation_status = 'ok' AND pbr > 0) AS calculable,
        COUNT(*) FILTER (WHERE calculation_status = 'ok' AND is_below_pbr_threshold) AS below_book,
        below_book::DOUBLE / NULLIF(calculable, 0) AS below_book_ratio
    FROM pbr_research_base
    GROUP BY sector
    HAVING calculable > 0
    ORDER BY below_book_ratio DESC, calculable DESC
    '''
).df()
display(sector)
sector.set_index('sector')['below_book_ratio'].sort_values().plot.barh(figsize=(8, 10))
plt.xlabel('PBR below threshold ratio')
plt.tight_layout()

## 4. Candidate review table — H3 / H4
これは候補のEvidence表です。値の良否を総合スコア化せず、不足項目も同時に表示します。

In [ ]:
candidates = connection.execute(
    '''
    SELECT *
    FROM pbr_candidate_evidence
    ORDER BY missing_evidence = '' DESC, roe_proxy DESC NULLS LAST, pbr
    '''
).df()
candidate_columns = [
    'security_key', 'company_name', 'market_name', 'sector_33_name',
    'as_of_date', 'pbr', 'roe_proxy', 'equity_ratio', 'operating_margin',
    'operating_cash_flow', 'free_cash_flow_proxy', 'financial_disclosed_date',
    'financial_period_end', 'financial_period_age_days', 'is_stale_financial',
    'missing_evidence',
]
display(candidates[candidate_columns].head(30))
display(candidates['missing_evidence'].value_counts(dropna=False).head(20).to_frame('rows'))

## 5. Optional descriptive follow-up

3・6・12か月forward returnは、上場廃止を含む過去ユニバース、配当、売買可能性、基準日重複を十分に扱えないため、このPilotの必須成果物にはしません。実装する場合も**配当を含まないprice returnの記述統計**とし、投資戦略のバックテストとは呼びません。

## 6. Findings / limitations / promotion candidates

実行後、次をH1〜H4ごとに記録します。

- **Observed finding:** 集計から直接確認できた事実
- **Interpretation:** 事実に対する暫定解釈
- **Counter evidence:** 解釈に反する証拠
- **Unknown:** 現行データでは判断できないこと
- **Promotion candidate:** 継続監視またはManual Assessmentへ渡す項目

最低限の制約は、先読み防止、最新銘柄マスタによる生存者バイアス、配当を含まない価格リターン、B/S詳細・カタリスト未取得です。Manual Assessmentへ渡す銘柄は5〜10件に絞り、仮説・反証条件・必要Evidenceを別途記録します。